# Packages

In [2]:
# core
import pandas as pd
import numpy as np
import polars as pl
import math
import requests

# scipy / stats
import scipy.stats as stats
from scipy.optimize import minimize

# visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# statsmodels
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.tsa.deterministic import DeterministicProcess, CalendarFourier
from statsmodels.graphics.tsaplots import plot_pacf

# sklearn
from sklearn.model_selection import (
    GroupKFold, KFold, train_test_split, cross_val_score, TimeSeriesSplit
)
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.metrics import (
    mean_squared_error, r2_score, make_scorer, accuracy_score,
    median_absolute_error, classification_report, mean_absolute_error,
    roc_auc_score, roc_curve
)
from sklearn.pipeline import make_pipeline
from sklearn.decomposition import PCA
from sklearn.ensemble import (
    RandomForestRegressor, GradientBoostingRegressor,
    StackingRegressor, RandomForestClassifier
)

# gradient boosting
import xgboost as xgb
import lightgbm as lgb
import catboost as cb

# deep learning
import torch
import torchvision
import torchaudio

# optimization
import optuna

# pandas display settings
pd.set_option('display.max_columns', 500)
pd.set_option('display.max_rows', 500)
pd.set_option('display.float_format', lambda x: "%.4f" % x)
pd.options.plotting.backend = "plotly"

# plotting styles
plt.style.use('ggplot')
sns.set_style('darkgrid')


# Data

In [3]:
train = pd.read_csv("./data/train.csv")
train.head()

,stock_id,date_id,seconds_in_bucket,imbalance_size,imbalance_buy_sell_flag,reference_price,matched_size,far_price,near_price,bid_price,bid_size,ask_price,ask_size,wap,target,time_id,row_id
0,0,0,0,3180602.6900,1,0.9998,13380276.6400,NaN,NaN,0.9998,60651.5000,1.0000,8493.0300,1.0000,-3.0297,0,0_0_0
1,1,0,0,166603.9100,-1,0.9999,1642214.2500,NaN,NaN,0.9999,3233.0400,1.0007,20605.0900,1.0000,-5.5200,0,0_0_1
2,2,0,0,302879.8700,-1,0.9996,1819368.0300,NaN,NaN,0.9994,37956.0000,1.0003,18995.0000,1.0000,-8.3900,0,0_0_2
3,3,0,0,11917682.2700,-1,1.0002,18389745.6200,NaN,NaN,1.0000,2324.9000,1.0002,479032.4000,1.0000,-4.0102,0,0_0_3
4,4,0,0,447549.9600,-1,0.9995,17860614.9500,NaN,NaN,0.9994,16485.5400,1.0000,434.1000,1.0000,-7.3498,0,0_0_4


In [4]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 5237980 entries, 0 to 5237979
Data columns (total 17 columns):
 #   Column                   Dtype  
---  ------                   -----  
 0   stock_id                 int64  
 1   date_id                  int64  
 2   seconds_in_bucket        int64  
 3   imbalance_size           float64
 4   imbalance_buy_sell_flag  int64  
 5   reference_price          float64
 6   matched_size             float64
 7   far_price                float64
 8   near_price               float64
 9   bid_price                float64
 10  bid_size                 float64
 11  ask_price                float64
 12  ask_size                 float64
 13  wap                      float64
 14  target                   float64
 15  time_id                  int64  
 16  row_id                   str    
dtypes: float64(11), int64(5), str(1)
memory usage: 679.4 MB


In [6]:
train["stock_id"].nunique()

200